# Recalculating Word Translation Entropy From Multiple Studies:

## [CRITT Academy](https://github.com/Critt-Kent/CRITT-Academy) Module 01: Lesson 07

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/02_Summary_Info.ipynb) | [This video linked here](https://vimeo.com/1201603642) explains how to use this code notebook in Google Colab 🤠

### What You Will Do In This Lesson

Entropy-related values are computed on a word level for each study. They can be re-computes for several studies, increasing the accuracy of the values, or on a fraction of a study. 

1. a) Install and import all necessary Python libraries and then b) fetch necessary data from the CRITT TPR-DB

2. Recompute entropy-related values on a word level for ST tables across several studies:
   - `HTra`: word translation Entropy
   - `HTraN`: normalized `HTra`
   - `InfT`: word translation information $log(1/P(\text{translation}))$
   - `InfS`: grouping probability: Probability to cluster ST words for the translation
     
3. Carry over the word-based values to the segment level to compute:
    - `HTraN`: average normalized `HTraN`
    - `HTot`: sum over word-based `HTra` values values
    - `InfT`: mean word-based `InfT` values
    - `InfS`: mean word-based `InfS` values
      
4. Carry over the word-based `HTra`, `HTraN`, `InfS`, and `InfT` to other kinds of units, e.g., `PUs`, `AUs` 

### First time working with a CRITT Academy code notebook?

If you haven't yet gone through the [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR_DB”](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB.ipynb), you should do that first (this will help you understand Step 1 much better).

## Step 1a. Import all necessary Python libraries


In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "numpy": "2.4.0",
    "pandas": "3.0.0",
    "scipy": "1.17.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.0",
    "tprdb-utilities": "0.7.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tprdb_utilities

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")

    try:
        # The %pip magic ensures installation happens in the active Jupyter kernel
        %pip install "numpy>=2.4.0" "pandas>=3.0.0" "scipy>=1.17.0" "matplotlib>=3.10.0" "seaborn>=0.13.0" "tprdb-utilities>=0.7.0"

        print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")
    except Exception as e:
        print(f"❌ An error occurred during installation: {e}")
        print("If using Google Colab, you may just have to restart your runtime now to use the newly installed packages.")

In [ ]:
# Now, import the libraries

try:
    # Standard Python library imports
    import glob
    import os
    import re

    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

    # TPR-DB utilities import
    from tprdb_utilities import fetch_TPRDB_tables, read_TPRDB_tables

    print("✅ All imports successful!")

except ImportError as e:
    print(f"❌ An error occurred during imports: {e}")
    print("Please ensure all dependencies are installed and the kernel is restarted if necessary.")

## Step 1b: Fetch Tables Data


In [ ]:
# Commenting/Uncommenting Code: ⌘ + / (Mac) or Ctrl + / (Windows/Linux)

# 🫵 If NOT using Google Colab:
# 1) Change what's in the quotes below ("./") to point to where you want your mothership clone to be saved.
# 2) Then, run this code cell
mothership_clone_location = "./"

# 🫵 If using Google Colab:
# 1) Comment out the above line of code
# 2) Uncomment the lines of code below
# 3) Then, run this code cell

# from google.colab import drive
# drive.mount('/content/drive')
# mothership_clone_location = "/content/drive/MyDrive/"

In [ ]:
# fetch data tables (notice we are using the `mothership_clone_location` variable we defined in the cell above)

# these studies use (English) Multiling texts translation into Chinese
studiesList=["RUC17", "RUCMT17", "STML18bolt", "HNUJml", "STC17bolt", "ZHMT19", "MS12"]

fetch_TPRDB_tables(
    path=mothership_clone_location,
    studies=studiesList,
    extensions=["ss"],
    public=True,
)

## Step 2: Read ST data from different studies into a DataFrame

When the studies have the same source text and identical language pairs, we can re-compute the entropy values based on the joint dataset. 


In [ ]:
# read Source Token tables (ST) from several different studies
STdf = read_TPRDB_tables(
    path=f'{mothership_clone_location}/tprdb-mothership-clone',
    user='PUBLIC',
    studies=studiesList,
    extension="st",
    verbose=1,
)

In [ ]:
# check distribution of tasks in the dataframe

sns.countplot(STdf, x="Task")